# Parquet and Columnar Storage

The institute has 10 years of hourly temperature data. That is roughly 87,000 rows per year, and we have data from multiple stations. Reading the CSV takes a noticeable pause. The file is large. And most of the time, we only need two or three columns.

There must be a better way. There is.

In this notebook we will meet **Parquet**, a columnar storage format that has become the de facto standard for analytical data. It is smaller, faster to read, and lets you load only the columns you actually need. Once you understand why it works, you will never want to store large analytical datasets as CSV again.

## Setup

In [ ]:
%pip install -q pyarrow

In [ ]:
import pandas as pd
import os
import time

## Part 1: Loading the Historical Data

We have a CSV file with ~50,000 rows of hourly weather readings from a single station spanning several years. Let's load it and see what we are working with.

In [ ]:
start = time.time()
df = pd.read_csv("../data/temperature_historical.csv")
csv_read_time = time.time() - start

print(f"Shape: {df.shape}")
print(f"CSV read time: {csv_read_time:.3f} seconds")
df.head()

In [ ]:
df.dtypes

In [ ]:
csv_size = os.path.getsize("../data/temperature_historical.csv")
print(f"CSV file size: {csv_size:,} bytes ({csv_size / 1024 / 1024:.2f} MB)")

50,000 rows of hourly data. The CSV file is a few megabytes. For a single station this is manageable, but imagine scaling to hundreds of stations, each with decades of data. CSV starts to creak under that kind of weight.

The problems with CSV for large analytical data:

1. **No compression** -- every character is stored as text
2. **No types** -- everything is a string that must be parsed on read
3. **Row-oriented** -- to read one column, you must scan the entire file
4. **No metadata** -- the reader has to guess types, count rows, etc.

Let's understand that third point, because it is the fundamental insight behind columnar formats.

## Part 2: Row-Oriented vs Columnar Storage

Imagine you have a table with 4 columns and millions of rows:

```
station_id | date       | hour | temp_c
WS001      | 2018-01-01 | 0    | -0.1
WS001      | 2018-01-01 | 1    | -5.8
WS001      | 2018-01-01 | 2    | -4.5
...
```

### Row-oriented (CSV, most databases)

Data is stored row by row:

```
[WS001, 2018-01-01, 0, -0.1] [WS001, 2018-01-01, 1, -5.8] [WS001, 2018-01-01, 2, -4.5] ...
```

If you want just the `temp_c` column, you still have to read every byte of the file and skip over `station_id`, `date`, and `hour` for every single row. You are reading 4x more data than you need.

### Columnar (Parquet, ORC)

Data is stored column by column:

```
station_id: [WS001, WS001, WS001, ...]
date:       [2018-01-01, 2018-01-01, 2018-01-01, ...]
hour:       [0, 1, 2, ...]
temp_c:     [-0.1, -5.8, -4.5, ...]
```

Now if you want just `temp_c`, you read only that column's data. The other columns are not touched at all. This is called **column pruning** or **projection pushdown**.

There is a second benefit: within a single column, values tend to be very similar (all the same type, similar ranges). This makes compression far more effective. A column of station IDs that is always "WS001" compresses to almost nothing. A column of temperatures in a narrow range compresses beautifully.

The result: columnar files are typically **3-10x smaller** than CSV, and reading specific columns is **much faster**.

## Part 3: Writing Parquet

Converting a DataFrame to Parquet is a one-liner. pandas uses `pyarrow` under the hood.

In [ ]:
# Write the DataFrame to Parquet
parquet_path = "/tmp/temperature_historical.parquet"
df.to_parquet(parquet_path, index=False)

In [ ]:
parquet_size = os.path.getsize(parquet_path)
print(f"CSV file size:     {csv_size:,} bytes ({csv_size / 1024 / 1024:.2f} MB)")
print(f"Parquet file size: {parquet_size:,} bytes ({parquet_size / 1024 / 1024:.2f} MB)")
print(f"\nCompression ratio: {csv_size / parquet_size:.1f}x smaller")

The Parquet file should be significantly smaller than the CSV. The exact ratio depends on the data, but 3-5x is typical for numerical data like this. For data with lots of repeated strings, the savings can be even more dramatic.

## Part 4: Reading Parquet -- The Head-to-Head

Now for the real test. Let's compare read times.

In [ ]:
# Time reading the full CSV
start = time.time()
df_csv = pd.read_csv("../data/temperature_historical.csv")
csv_full_time = time.time() - start
print(f"CSV full read:     {csv_full_time:.4f}s")

# Time reading the full Parquet
start = time.time()
df_parquet = pd.read_parquet(parquet_path)
parquet_full_time = time.time() - start
print(f"Parquet full read: {parquet_full_time:.4f}s")

print(f"\nParquet is {csv_full_time / parquet_full_time:.1f}x faster for full read")

Even reading all columns, Parquet is faster because it does not need to parse text into numbers -- the data is already stored in binary format with type information.

But the real advantage shows when you only need a few columns.

## Part 5: Column Pruning

This is where Parquet really shines. Suppose we only want the date and temperature columns. With CSV, we still have to read the entire file:

In [ ]:
# CSV: read everything, then select columns
start = time.time()
df_csv_subset = pd.read_csv("../data/temperature_historical.csv", usecols=["date", "temp_c"])
csv_subset_time = time.time() - start
print(f"CSV (2 columns):     {csv_subset_time:.4f}s")

# Parquet: only read the columns we need
start = time.time()
df_pq_subset = pd.read_parquet(parquet_path, columns=["date", "temp_c"])
parquet_subset_time = time.time() - start
print(f"Parquet (2 columns): {parquet_subset_time:.4f}s")

print(f"\nParquet is {csv_subset_time / parquet_subset_time:.1f}x faster for column selection")

In [ ]:
df_pq_subset.head()

When you specify `columns=[...]` with Parquet, it physically only reads those columns from disk. The other columns are never loaded into memory. With CSV, `usecols` still has to scan through every row to find the column positions.

For wide datasets (50+ columns) where you only need a handful, the performance difference is dramatic.

## Part 6: Parquet Metadata

One of Parquet's advantages is that it stores metadata alongside the data. Let's inspect it.

In [ ]:
import pyarrow.parquet as pq

parquet_file = pq.ParquetFile(parquet_path)

print("Schema:")
print(parquet_file.schema_arrow)
print(f"\nNumber of row groups: {parquet_file.metadata.num_row_groups}")
print(f"Number of rows: {parquet_file.metadata.num_rows}")
print(f"Number of columns: {parquet_file.metadata.num_columns}")

The schema tells us the exact data type of each column before we read a single row. The row group information tells us how many rows there are and how the data is physically chunked.

This metadata is why Parquet does not need to guess types. When pandas reads a CSV, it has to scan through values to infer that a column is integer vs float vs string. Parquet already knows.

In [ ]:
# Row group details
for i in range(parquet_file.metadata.num_row_groups):
    rg = parquet_file.metadata.row_group(i)
    print(f"Row group {i}: {rg.num_rows} rows, {rg.total_byte_size:,} bytes")

### Compression options

Parquet supports several compression algorithms. The default in pandas/pyarrow is `snappy`, which offers a good balance of speed and compression. You can also use `gzip` (slower but smaller) or `zstd` (best of both worlds, but requires extra libraries).

In [ ]:
# Compare compression methods
for compression in ["snappy", "gzip", None]:
    path = f"/tmp/temp_{compression}.parquet"
    df.to_parquet(path, compression=compression, index=False)
    size = os.path.getsize(path)
    label = compression if compression else "none"
    print(f"{label:>10}: {size:>10,} bytes ({size/1024/1024:.2f} MB)")

print(f"{'CSV':>10}: {csv_size:>10,} bytes ({csv_size/1024/1024:.2f} MB)")

Even uncompressed Parquet is smaller than CSV, because it stores numbers in binary rather than text. Compression makes it smaller still.

## Part 7: Type Preservation

Another quiet advantage of Parquet: types survive the round trip. With CSV, you write integers and get back... strings that pandas has to guess are integers. Dates become strings. Booleans become strings.

With Parquet, what you write is what you get back.

In [ ]:
# Check types from CSV read
print("Types from CSV:")
print(df_csv.dtypes)
print()

# Check types from Parquet read
print("Types from Parquet:")
df_from_parquet = pd.read_parquet(parquet_path)
print(df_from_parquet.dtypes)

Notice how the `date` column from CSV is an `object` (string), while from Parquet it retains whatever type it was saved as. This means fewer type conversion headaches when you use Parquet as your storage format.

## Part 8: When to Use What

CSV and Parquet are not competitors -- they serve different purposes.

**Use CSV when:**
- You need humans to read/edit the file
- You are exchanging small amounts of data with non-technical users
- You need maximum compatibility (every tool reads CSV)
- The data is small (under a few thousand rows)

**Use Parquet when:**
- The data is for analytical processing, not human reading
- The dataset is large (thousands to billions of rows)
- You need fast reads, especially of specific columns
- You want type preservation without manual parsing
- You are building a data pipeline or data lake

In practice, a common pattern is: ingest data as CSV (because that is what the source provides), convert to Parquet for storage and processing, and export results as CSV (because that is what stakeholders want).

## The Numbers

Let's put together a summary comparison.

In [ ]:
comparison = pd.DataFrame({
    "Metric": ["File size (bytes)", "Full read time (s)", "2-column read time (s)"],
    "CSV": [csv_size, round(csv_full_time, 4), round(csv_subset_time, 4)],
    "Parquet": [parquet_size, round(parquet_full_time, 4), round(parquet_subset_time, 4)],
})
comparison["Ratio (CSV/Parquet)"] = (comparison["CSV"] / comparison["Parquet"]).round(1)
comparison

---

## Exercises

### Exercise 1: Convert and compare

Read `weather_stations.csv` into a DataFrame. Write it to Parquet. Compare the file sizes of the CSV and Parquet versions. What compression ratio do you get?

In [ ]:
# Your code here


### Exercise 2: Column pruning in practice

Using the Parquet file you created in Exercise 1:

1. Read only the `station_name` and `avg_temp_c` columns from the Parquet file
2. Find the station with the highest average temperature
3. Time this operation and compare it to doing the same thing from CSV

In [ ]:
# Your code here


### Exercise 3: Compression showdown

Write the historical temperature data to three Parquet files using different compression:
- `snappy` (the default)
- `gzip`
- No compression (`None`)

For each, measure the file size and the time to read the file back. Which compression gives the best trade-off between size and speed?

In [ ]:
# Your code here


### Exercise 4: Schema inspection

Using `pyarrow.parquet`, inspect the schema of the historical temperature Parquet file. Answer these questions:

1. What data types does Parquet assign to each column?
2. How many row groups does the file contain?
3. How many total bytes are in the first row group?

Hint: use `pq.ParquetFile()` and explore the `.schema_arrow` and `.metadata` attributes.

In [ ]:
# Your code here
